# Tech Challenge Fiap - Fase 1

## Case NPS Preditivo
Prever satisfação do cliente (NPS) antes da aplicação da pesquisa

## Fase 3 - Data Preparation

Aqui é onde vamos preparar os dados para a fase 4 do modelo. 

Agora que já descobrimos as variáveis mais correlacionadas: complaints_count (0.45) e delivery_delay_days (0.40) através da Análise Exploratória de Dados. 

Nessa fase avançaremos em:
- Remover colunas irrelevantes e com leakage.
- Aplicar One-Hot Coding na variável texto.
- Separar o que é Feature e o que é Target.
- Dividir dados em Treino e teste.  

In [15]:
# Libraries 

import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# Loading .csv
# Creating Target variable 

df = pd.read_csv("desafio_nps_fase_1.csv")

df["nps_detrator"] = (df['nps_score'] <= 6).astype(int)



### Removendo variáveis

Removendo variáveis sem relevância e as proibidas por leakage (csat_internal_score, repeat_purchase_30d). 
A variável nps_score foi removida pois como comentado na na fase 1 (Business Understanding) não temos essa variável até o momento do fim da jornada do cliente. Por esse motivo caracterizamos essa variável como leakage. 

Jornada do cliente: pedido realizado → pagamento → separação → envio → entrega → pesquisa NPS 

In [16]:
colunas_remover = ['customer_id', 'order_id', 'csat_internal_score', 'repeat_purchase_30d', 'nps_score']

df = df.drop(colunas_remover, axis=1)
df.head()

,customer_age,customer_region,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,nps_detrator
0,63,Nordeste,14,139.73,4,39.35,4,2,2,55.53,3,0,4,3,0
1,20,Sul,1,458.95,2,9.51,10,6,4,28.23,3,0,10,3,1
2,46,Nordeste,111,507.06,5,42.82,6,6,1,40.99,1,4,5,7,1
3,52,Centro-Oeste,117,302.19,2,19.58,9,5,2,35.24,3,1,11,4,1
4,56,Norte,50,253.06,1,29.37,11,13,1,39.32,1,1,0,3,0


### Tratando variáveis categóricas

O modelo de Machine Learning não entende texto como "Sudeste" ou "Sul". Precisamos transformar essa variável em números com **One-Hot Enconding**.

**O que é One-Hot Encoding?**  
Transforma uma coluna categórica em várias colunas binárias (0 ou 1).

**Por que usar One-Hot Encoding?**  
Como não temos **viés regional**, significa que provavelmente a variável customer_region não será uma feature importante para o modelo. Mas devemos incluir no modelo e deixá-lo decidir. 

**Se tivesse viés regional?**   
O modelo daria mais peso para uma região. Por exemplo, Norte com 90% e Sul com 50%, o modelo aprenderia que a região é um fator preditivo forte. A empresa tomaria ações regionais.  


Como não há viés regional, o modelo provavelmente vai aprender que customer_region tem baixo poder preditivo. 


In [17]:
df = pd.get_dummies(df, columns=['customer_region'], drop_first=True)

df.columns

Index(['customer_age', 'customer_tenure_months', 'order_value',
       'items_quantity', 'discount_value', 'payment_installments',
       'delivery_time_days', 'delivery_delay_days', 'freight_value',
       'delivery_attempts', 'customer_service_contacts',
       'resolution_time_days', 'complaints_count', 'nps_detrator',
       'customer_region_Nordeste', 'customer_region_Norte',
       'customer_region_Sudeste', 'customer_region_Sul'],
      dtype='str')

In [18]:
df.head(10)

,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,nps_detrator,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul
0,63,14,139.73,4,39.35,4,2,2,55.53,3,0,4,3,0,True,False,False,False
1,20,1,458.95,2,9.51,10,6,4,28.23,3,0,10,3,1,False,False,False,True
2,46,111,507.06,5,42.82,6,6,1,40.99,1,4,5,7,1,True,False,False,False
3,52,117,302.19,2,19.58,9,5,2,35.24,3,1,11,4,1,False,False,False,False
4,56,50,253.06,1,29.37,11,13,1,39.32,1,1,0,3,0,False,True,False,False
5,35,75,568.76,6,36.58,3,4,5,41.82,2,2,3,5,1,False,False,True,False
6,37,68,41.29,3,99.62,6,8,3,35.83,3,3,4,6,1,False,False,True,False
7,60,37,428.76,4,29.54,10,11,5,44.50,1,0,2,2,1,False,False,False,True
8,40,60,121.56,3,91.95,6,6,3,24.88,2,1,9,3,0,False,False,False,True
9,51,70,411.01,6,37.47,3,9,2,30.59,1,0,7,2,1,False,False,True,False


### Separando Features e Target

X → todas as colunas que o modelo vai usar para aprender (features)  
y → a coluna que o modelo vai prever (target)  

Features = Todas as colunas do dataframe df menos a coluna nps_detrator — o nps_score já foi removido na etapa anterior por risco de leakage. 

Target = nps_detrator  


In [19]:
X = df.drop(columns=['nps_detrator'])
y = df['nps_detrator']

print(X.shape)
print(y.shape)

(2500, 17)
(2500,)


### Dividindo Dataset em Treino e Teste 

Precisamos dividir nossos dados em dois grupos: 

Treino → o modelo aprende com esses dados  
Teste → o modelo é avaliado com dados que nunca viu

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(2000, 17)
(500, 17)


Treino → 2000 linhas (80% dos dados): O modelo aprende com isso  
Teste → 500 linhas (20% dos dados): O modelo será avaliado com isso

A proporção 80/20 é a mais comum em datasets de tamanho médio, garante dados suficientes para treino sem comprometer a avaliação.   
O random_state=42 garante reprodutibilidade, ou seja, a divisão será sempre a mesma em qualquer execução.

Preparando os dados para a fase 4 - Data Modeling

In [21]:
# Preparando os dados para fase 4- Data Modeling

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

Agora que nossos dados estão prontos, o próximo passo é avançar para a fase 4, ou a fase da modelagem.